# Задание #1. Классификация морфологии галактик

Современные широкоугольные телескопы, такие как [DESI](https://en.wikipedia.org/wiki/Dark_Energy_Spectroscopic_Instrument), ежедневно делают миллионы снимков глубокого космоса. Объём данных настолько велик, что астрономы физически не успевают классифицировать каждую галактику вручную.

Вы — ведущий инженер по ИИ в международном космическом агентстве. Ваша задача — создать автономную систему, которая сможет с высокой точностью определять морфологический тип галактики по её изображению. Это критически важно для понимания того, как Вселенная эволюционировала за миллиарды лет. Ошибка в классификации может привести к неверным выводам о распределении темной материи и скорости расширения космоса.

## Данные
В вашем распоряжении находится набор данных, состоящий из `155,951` изображения галактик, собранных и размеченных волонтерами проекта [Galaxy Zoo](https://data.galaxyzoo.org/). Данный набор поделен на 8 классов, каждый из которых отражает определенную морфологию. 

<table>
    <thead>
        <tr>
            <th align="left">ID класса</th>
            <th align="left">Имя класса</th>
            <th align="left">Описание</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td align="left">0</td>
            <td align="left">Круглая эллиптическая</td>
            <td align="left">Гладкая галактика, полностью круглой формы.</td>
        </tr>
        <tr>
            <td align="left">1</td>
            <td align="left">Полуэллиптическая</td>
            <td align="left">Гладкая галактика, слегка удлиненная.</td>
        </tr>
        <tr>
            <td align="left">2</td>
            <td align="left">Сигарообразная эллиптическая</td>
            <td align="left">Гладкая галактика, сильно удлиненная</td>
        </tr>
        <tr>
            <td align="left">3</td>
            <td align="left">Спиральная, видимая с ребра</td>
            <td align="left">Спиральная галактика, рассматриваемая сбоку (диск виден).</td>
        </tr>
        <tr>
            <td align="left">4</td>
            <td align="left">Спиральная с перемычкой</td>
            <td align="left">Спиральная галактика с видимой центральной перемычкой</td>
        </tr>
        <tr>
            <td align="left">5</td>
            <td align="left">Спиральная без перемычки</td>
            <td align="left">Спиральная галактика без видимой центральной перемычки</td>
        </tr>
        <tr>
            <td align="left">6</td>
            <td align="left">Неправильная</td>
            <td align="left">Галактика без определённой формы или с нарушенной структурой.</td>
        </tr>
        <tr>
            <td align="left">7</td>
            <td align="left">Смешанная</td>
            <td align="left">Совмещение нескольких видов.</td>
        </tr>
    </tbody>
</table>

### Структура данных
Дано два поля:
- `image`: изображение галактики в `PIL.Image.Image`, исходный размер — `424x424` пикселя.
- `label`: целочисленное значение, содержащее номер класса `(0–7)`

Набор делится на:
- тренировочные данные: `~99.8k` изображений
- валидационные данные: `~25k` изображений
- тестовые данные: `~31.2k` изображений

Классы несбалансированы, что следует учитывать при выборе функции потерь и стратегии обучения. Соотношение классов в тренировочной выборке:
<table>
    <thead>
        <tr>
            <th align="left">ID класса</th>
            <th align="left">Доля</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td align="left">0</td>
            <td align="left">21.7%</td>
        </tr>
        <tr>
            <td align="left">1</td>
            <td align="left">25.2%</td>
        </tr>
        <tr>
            <td align="left">2</td>
            <td align="left">4.8%</td>
        </tr>
        <tr>
            <td align="left">3</td>
            <td align="left">10.5%</td>
        </tr>
        <tr>
            <td align="left">4</td>
            <td align="left">13.1%</td>
        </tr>
        <tr>
            <td align="left">5</td>
            <td align="left">19.7%</td>
        </tr>
        <tr>
            <td align="left">6</td>
            <td align="left">3.7%</td>
        </tr>
        <tr>
            <td align="left">7</td>
            <td align="left">1.4%</td>
        </tr>
    </tbody>
</table>



### Загрузка данных
Обратитесь к администрации и уточните, где можно загрузить набор, если нет доступа к ресурсу HuggingFace.

In [ ]:
import polars as pl

splits = {'train': 'data/train-*.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pl.read_parquet("hf://datasets/mrJordi0/galaxy-zoo-dataset/" + splits["train"])

In [ ]:
from datasets import load_dataset

ds = load_dataset("mrJordi0/galaxy-zoo-dataset")

## Метрики
В этом задании важна аккуратная работа с недопредставленными классами, поэтому основной метрикой для оценки будет [$Macro \space F_{1}$](https://en.wikipedia.org/wiki/F-score) ($K=8$):
$$
F_{1, macro} = \frac{1}{K} \cdot \sum_{i=1}^{K} F_{1,i}
$$
</br>
где:
</br>
$$
F_{1} = 2 \cdot \frac{precision \cdot recall}{precision + recall} = \frac{2TP}{2TP+FP+FN}
$$
</br>
Вспомогательной метрикой будем считать `Balanced Accuracy`. Для подсчета `Balanced Accuracy` необходимо вычислить стандартную `Accuracy` для каждого класса:
</br></br>
$$
Accuracy = \frac{TP+TN}{TP+TN+FP+FN}
$$
</br>
и взять среднее значение.
</br></br>
Кроме этого будет учитываться **время обработки** одного изображения и **конечный вес модели**.

# Ограничение
Запрещено:
- дообучать моделы на набораx данных кроме представленного;
- обучать модели на тестовых наборах;
- применять ансамбли моделей.

Работы, которые нарушили данные ограничения, **рассмотрены не будут**.

# Baseline
Ниже приведен базовый стенд для тестирования модели, обученной с помощью `PyTorch`

In [ ]:
import os
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from datasets import load_dataset
from sklearn.metrics import f1_score, balanced_accuracy_score
import polars as pl

class GalaxyValidator:
    def __init__(self, model_path, test_limit=None):
        self.model_path = model_path
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.test_limit = test_limit
        self.img_size = 224
        self.batch_size = 1

        self.transform = transforms.Compose(
            [
                transforms.Resize((self.img_size, self.img_size)),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
            ]
        )

    def load_test_data(self):
        ds = load_dataset("mrJordi0/galaxy-zoo-dataset", split="test")
        if self.test_limit:
            ds = ds.select(range(self.test_limit))

        return ds

    def evaluate(self, model):
        test_ds = self.load_test_data()
        all_preds = []
        all_labels = []
        latencies = []

        model.eval()
        model.to(self.device)

        print(f"Starting evaluation on {len(test_ds)} images...")

        with torch.no_grad():
            for item in test_ds:
                image = item["image"].convert("RGB")
                label = item["label"]

                input_tensor = self.transform(image).unsqueeze(0).to(self.device)

                start_time = time.perf_counter()
                output = model(input_tensor)
                end_time = time.perf_counter()

                pred = torch.argmax(output, dim=1).item()

                all_preds.append(pred)
                all_labels.append(label)
                latencies.append(end_time - start_time)

        metrics = {
            "f1_macro": f1_score(all_labels, all_preds, average="macro"),
            "balanced_accuracy": balanced_accuracy_score(all_labels, all_preds),
            "avg_latency_ms": np.mean(latencies) * 1000,
            "model_size_mb": os.path.getsize(self.model_path) / (1024 * 1024),
        }

        return metrics

    def run_check(self, model_instance):
        try:
            state_dict = torch.load(self.model_path, map_location=self.device)
            model_instance.load_state_dict(state_dict)

            results = self.evaluate(model_instance)

            print("\n" + "=" * 30)
            print("FINAL SCORECARD")
            print("=" * 30)
            print(f"F1-Macro Score:      {results['f1_macro']:.4f}")
            print(f"Balanced Accuracy:   {results['balanced_accuracy']:.4f}")
            print(f"Avg Inference Time:  {results['avg_latency_ms']:.2f} ms")
            print(f"Model Weight:        {results['model_size_mb']:.2f} MB")
            print("=" * 30)

            if results["model_size_mb"] > 100:
                print("WARNING: Model exceeds 100MB limit!")
            if results["avg_latency_ms"] > 50:
                print("WARNING: Inference is too slow (>50ms)!")

        except Exception as e:
            print(f"Evaluation Failed: {e}")